In [ ]:
checkpoint_path: str='./lightning_logs/Everything_TS=4/1488392/last.ckpt' # can be a glob pattern, in which case the last checkpoint is chosen
n_flow_through_times: int=2 # number of flow through times to simulate to determine long term behavior
n_sim_pred_samples: int=5 # number of samples to use for the simulation
fast_dataloaders: bool=True # whether to use fast dataloaders (don't overwhelm memory with large models)
random_IC: bool=False # whether to use a random IC for the simulation
seed: int=0 # seed for RNG

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline
import torch
import numpy as np
import matplotlib.pyplot as plt
import pytorch_lightning as L
import torch.nn.functional as F
import torch.multiprocessing
torch.multiprocessing.set_sharing_strategy('file_system')
torch.set_float32_matmul_precision('medium')
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

from utils import *
from lightning_utils import *
import JHTDB_sim_op
from JHTDB_sim_op import POU_NetSimulator, PPOU_NetSimulator

import model_agnostic_BNN # the script is now fully compatible with the current model
from model_agnostic_BNN import PredSamplingWrapper

L.seed_everything(seed)

In [ ]:
from JHTDB_data_loading import JHTDBDataModule

# long horizon is explicit for legacy reasons to patch a previous bug
data_module = JHTDBDataModule.load_from_checkpoint(checkpoint_path, long_horizon=400, fast_dataloaders=fast_dataloaders)

In [ ]:
# load the full simulation tensor
real_channel_flow = data_module.load_full_dataset_field()
print(f'real_channel_flow:')
tshow(real_channel_flow) # show size in GBs
#print(f'real_channel_flow size: {np.prod(real_channel_flow.shape)*4/1e9} GB')

### Compute the Characteristic Time:
GOTCHA: only works for MSE, different errors require different models! \
For dataset (with MSE): `t_c=127.52619893227192` `C=0.0019192068442813793`

In [ ]:
class CharacteristicTimeMSEModel:
    def __init__(self, field_tensor, n_windows=50):
        field_tensor = field_tensor.detach()
        window_size = field_tensor.shape[-1] - n_windows + 1
        print(f'window_size=field_tensor.shape[-1]-n_windows+1={window_size}')
        # ^ verified to work: 1/21/26

        # basic u0 MSE (for plotting later)
        self.u0 = field_tensor[...,0]
        self.u0_MSE = torch.vmap(torch.mean)((field_tensor.moveaxis(-1, 0)-self.u0)**2)

        MSEs = [] # MSEs[i] is the MSE of the u0 from time i to i+window_size
        from tqdm import tqdm
        for i in tqdm(range(field_tensor.shape[-1]-window_size+1)):
            u0 = field_tensor[...,i]
            time_window = field_tensor.moveaxis(-1, 0)[i:i+window_size]
            MSEs.append(torch.vmap(torch.mean)((time_window-u0)**2))
        MSEs = torch.stack(MSEs, dim=0)

        import scipy.optimize as optimize
        t = np.arange(window_size)
        def exp_error_residual(x):
            C, t_c = x # this is what the array means
            pred_MSE = C*(1-np.exp(-t/t_c))
            return (abs(MSEs-pred_MSE)).mean().item()
        opt_result = optimize.shgo(exp_error_residual, bounds=((1e-16, 100), (1, 500)))
        #opt_result = optimize.minimize(exp_error_residual, x0=np.random.uniform(1, 250, size=2))
        self.asymptotic_MSE, self.characteristic_time = opt_result.x
        print(f'{opt_result=}')
        print(f'characteristic_time=t_c={self.characteristic_time}, asymptotic_MSE=C={self.asymptotic_MSE}')

    def predict_MSE(self, t):
        return self.asymptotic_MSE*(1-np.exp(-t/self.characteristic_time))

    def plot_MSE_vs_pred(self, other_model=None, other_model_label='other'):
        print(f'characteristic_time=t_c={self.characteristic_time}, asymptotic_MSE=C={self.asymptotic_MSE}')
        t = np.arange(self.u0_MSE.shape[0])
        if other_model:
            plt.plot(other_model.u0_MSE, label=other_model_label+'.u0_MSE')
            plt.plot(other_model.predict_MSE(t), label=other_model_label+'.prediction')
        plt.plot(self.u0_MSE, label='$MSE(u_0,u_t)$')
        plt.plot(self.predict_MSE(t), label='prediction')
        plt.axvline(self.characteristic_time, color='k', linestyle='--', label=f't_c={self.characteristic_time}')
        plt.axhline(self.asymptotic_MSE, color='k', linestyle=':', label=f'C={self.asymptotic_MSE}')
        plt.legend()
        plt.title('MSE of u0 vs time')
        plt.show()

In [ ]:
mse_model = CharacteristicTimeMSEModel(real_channel_flow, n_windows=100)

In [ ]:
mse_model.plot_MSE_vs_pred()

In [ ]:
'''
NO_mse_model = CharacteristicTimeMSEModel(sim.flow_thru[0], n_windows=100)
NO_mse_model.plot_MSE_vs_pred()
mse_model.plot_MSE_vs_pred(other_model=NO_mse_model, other_model_label='NO_MSE')
''';